# Bronze Layer — Raw Ingestion

Stages the gzipped source files shipped in `data/` into the pipeline volume, then ingests the two
transactional files as-is into Delta tables with no transformation. Schema enforcement is
intentionally minimal: the goal is to preserve raw data exactly as received, so Silver
always has a full audit trail to recover from if cleaning logic changes.

On Databricks this notebook also creates the schema and volume if they don't exist yet, so
the pipeline needs no manual setup.

Geographic reference files (`post_codes.geojson`) are staged but intentionally not ingested
into Bronze. They are already clean, require no transformation, and are consumed directly as
raw assets by the Silver layer's zipcode backfill and by Gold.

In [ ]:
# Setup (identical in every pipeline notebook).
# On Databricks, catalog / schema / volume come from the job parameters defined in
# databricks.yml, or from the widgets at the top when run interactively. A blank
# catalog means "the workspace's default catalog". Nothing workspace-specific is
# hardcoded. Locally, all files live under data/output/.
import os
import sys

ON_DATABRICKS = "dbutils" in globals()  # dbutils exists only on Databricks

if ON_DATABRICKS:
    dbutils.widgets.text("catalog", "", "Catalog (blank = workspace default)")
    dbutils.widgets.text("schema", "amsterdam_rental", "Schema")
    dbutils.widgets.text("volume", "pipeline", "Volume")
    dbutils.widgets.text("repo_root", "", "Repo root (blank = auto-detect)")
    _nb_path = (
        dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    )
    _repo = dbutils.widgets.get("repo_root") or "/".join(_nb_path.split("/")[:-2])
    REPO_ROOT = _repo if _repo.startswith("/Workspace/") else "/Workspace" + _repo
else:
    REPO_ROOT = os.path.abspath("..")
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))

from amsterdam_rental.config import PipelineConfig, get_local_spark, resolve_catalog

if ON_DATABRICKS:
    cfg = PipelineConfig.for_databricks(
        REPO_ROOT,
        catalog=resolve_catalog(dbutils.widgets.get("catalog"), spark.catalog.currentCatalog()),
        schema=dbutils.widgets.get("schema"),
        volume=dbutils.widgets.get("volume"),
    )
else:
    spark = get_local_spark("amsterdam-rental-bronze")
    cfg = PipelineConfig.for_local(REPO_ROOT)
print(cfg.describe())

In [ ]:
from datetime import datetime, timezone

from pyspark.sql import functions as F

from amsterdam_rental.config import stage_sources

# Create the schema and volume on first run (needs USE CATALOG + CREATE SCHEMA on the
# catalog; on a workspace's default catalog these are usually granted).
if cfg.on_databricks:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{cfg.catalog}`.`{cfg.schema}`")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS `{cfg.catalog}`.`{cfg.schema}`.`{cfg.volume}`")

# Spark can't read workspace files on serverless, so decompress the repo's source files into
# the volume (locally: data/output/source). Overwrites on every run.
for name, path in stage_sources(cfg.data_dir, cfg.source_dir).items():
    print(f"staged {name:<20} -> {path}")

BRONZE_PATH = cfg.layer("bronze")
INGESTED_AT = datetime.now(timezone.utc).isoformat()

## Airbnb — bronze

In [ ]:
airbnb_raw = (
    spark.read.option("header", "true")
    .option("inferSchema", "false")  # keep everything as strings at bronze
    .csv(cfg.source("airbnb.csv"))
    .withColumn("_ingested_at", F.lit(INGESTED_AT))
    .withColumn("_source_file", F.lit("airbnb.csv"))
)

print(f"Airbnb rows: {airbnb_raw.count()}")
airbnb_raw.printSchema()
airbnb_raw.show(3, truncate=False)

In [ ]:
(airbnb_raw.write.format("delta").mode("overwrite").save(f"{BRONZE_PATH}/airbnb"))
print("Airbnb bronze written.")

## Rentals — bronze

In [ ]:
rentals_raw = (
    spark.read.option("multiline", "true")
    .json(cfg.source("rentals.json"))
    .withColumn("_ingested_at", F.lit(INGESTED_AT))
    .withColumn("_source_file", F.lit("rentals.json"))
)

print(f"Rentals rows: {rentals_raw.count()}")
rentals_raw.printSchema()
rentals_raw.show(3, truncate=True)

In [ ]:
(rentals_raw.write.format("delta").mode("overwrite").save(f"{BRONZE_PATH}/rentals"))
print("Rentals bronze written.")

## Validation checks

In [ ]:
airbnb_bronze = spark.read.format("delta").load(f"{BRONZE_PATH}/airbnb")
rentals_bronze = spark.read.format("delta").load(f"{BRONZE_PATH}/rentals")

assert airbnb_bronze.count() == 9913, "Unexpected Airbnb row count"
assert rentals_bronze.count() == 46722, "Unexpected Rentals row count"

# Geo reference files are intentionally not routed through Bronze
# (see README § Bronze); they're consumed as raw assets in Silver.

print("Bronze validation passed.")